# DRIFTX Gaussian workflow — Colab

Evidence-based Gaussian workflow. This notebook does not claim success until the cells complete.


In [ ]:
!git clone https://github.com/hitakshijoshi20072911/drift-3d.git /content/drift-3d 2>/dev/null || true
%cd /content/drift-3d
!pip install -e .


In [ ]:
!python -m driftx doctor --json
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')
print('VRAM MB:', round(torch.cuda.get_device_properties(0).total_memory/1024**2, 1) if torch.cuda.is_available() else 'not measured')


## C0–C5 sequence
Run the following in order, stopping on failure or OOM. Select an official Gaussian-capable DA3 checkpoint and record its license.


In [ ]:
# Example only; set MODEL to an official Gaussian-capable checkpoint before running.
MODEL = '/kaggle/input/da3-giant-1-1'
VIDEO = 'input/test3.mp4'
!python -m driftx benchmark --video $VIDEO --output outputs/cloud_c0 --model $MODEL --device cuda --sample-fps 1 --max-frames 4 --process-res 288 --reconstruction-mode gaussian
!python -m driftx benchmark --video $VIDEO --output outputs/cloud_c1 --model $MODEL --device cuda --sample-fps 1 --max-frames 8 --process-res 384 --reconstruction-mode gaussian


In [ ]:
from pathlib import Path
import json, zipfile
root=Path('outputs')
for report in sorted(root.glob('cloud_c*/run_report.json')):
    data=json.loads(report.read_text())
    print(report.parent, data.get('status'), data.get('error'), data.get('peak_gpu_memory_mb'))
with zipfile.ZipFile('driftx_gaussian_artifacts.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in root.glob('cloud_c*/*'):
        if p.is_file(): z.write(p)
print('Archive:', Path('driftx_gaussian_artifacts.zip').resolve())
